# Natural Frequency of Single Spring-Mass System

Checking what happens for single spring mass system with damping using the Julia code.

Good description at [iLearn Engg](https://www.ilearnengineering.com/mechanical/how-to-describe-and-analyse-damped-vibration-problems-involving-a-single-degree-of-freedom) and Book Boyce (2012)

Boyce, M.P., 2012. Rotor Dynamics, in: Gas Turbine Engineering Handbook. Elsevier, pp. 215–250. https://doi.org/10.1016/B978-0-12-383842-1.00005-6


### System Equations

$$
m \ddot{u} + c \dot{u} + k u = 0
$$

Standard harmonic solution form:
$$
u = \bar{u} \exp\left( -i \omega t + \alpha \right)
$$

Characteristic Equation
$$
(-\omega^2 m - i\omega c + k)\bar{u} = 0
$$

Critical damping 
$$
\frac{c_c^2}{4m^2} = \frac{k}{m} \\
\implies c_c = 2m \sqrt{\frac{k}{m}} = 2 \sqrt{k\,m}
$$

Damping factor 
$$
\xi = \frac{c}{c_c} = \frac{c}{2 \sqrt{k\,m}}
$$

Analytical damped natural frequency
$$
\omega_d = \omega_n \sqrt{1 - \xi^2}
$$

## Undamped System

In [1]:
using Pkg
Pkg.activate("/home/shagun/Acads/HydroElasticFEM.jl")

using HydroElasticFEM

PKG_ROOT = HydroElasticFEM.PKG_ROOT

  Activating project at `~/Acads/HydroElasticFEM.jl`


"/home/shagun/Acads/HydroElasticFEM.jl/"

In [2]:
using LinearAlgebra
using Roots: find_zero
using NLsolve

In [3]:
M = diagm([1.0, 4.0])

K = diagm([4.0, 100.0])

@show C = 0.1*K

Sol = M\K
λ = LinearAlgebra.eigvals(Sol)
V = LinearAlgebra.eigvecs(Sol)      

ωn = sqrt.(λ)
@show ωn
ξ = C ./ (2*sqrt.(K.*M))
@show ξ
ωd = ωn .* sqrt.(1 .- ξ.^2)
@show ωd

nothing

C = 0.1K = [0.4 0.0; 0.0 10.0]
ωn = [2.0, 5.0]
ξ = [0.1 NaN; NaN 0.25]
ωd = [1.98997487421324 NaN; NaN 4.841229182759271]


## Trying Complex Mass

Complex Mass
$$
(-\omega^2 M_\text{Full} + K)\bar{u} = 0 \\
M_\text{Full} = M + i \frac{C}{\omega}
$$

In [4]:
function damped_system(idx, ω, M, C, K)
    MFull = M + im*C/ω
    MFullReal = real.(MFull)

    Sol = MFull\K
    # Sol = MFullReal\K

    λ = LinearAlgebra.eigvals(Sol)
    V = LinearAlgebra.eigvecs(Sol)    
    
    @show sqrt.(λ)

    return λ, V, det(Sol)
end

idx = 2

ω = 2.0 + im*0
lIter = 0    
Δω = 1 
αRelax = 0.8

ωₒ = ω
detMat = 0*im

while (Δω > 1e-5)
    
    ωᵣ = αRelax * ω + (1 - αRelax) * ωₒ
    # ωᵣ = real(ωᵣ)    

    ωₒ = ω     
    detMat_old = detMat

    λ, V, detMat = damped_system(idx, ωᵣ, M, C, K)     
    ω = sqrt(λ[idx])    
    
    # Δω = abs((ω - ωₒ)/ωₒ)
    Δω = abs((detMat - detMat_old)/(detMat_old + 1e-10))
    lIter += 1
    @show lIter, ω, Δω
end

println("Final ω: ", ω) 
println("Analytical ωd: ", ωd) 
println("Difference: ", ω - diag(ωd)[idx])

nothing

sqrt.(λ) = ComplexF64[1.9708470956567796 - 0.195152320777687im, 3.561844588821655 - 1.7119110122732721im]
(lIter, ω, Δω) = (1, 3.561844588821655 - 1.7119110122732721im, 6.125638918316887e11)
sqrt.(λ) = ComplexF64[2.03647365821663 - 0.11100983331913013im, 4.7251679453883 - 1.8155253372388418im]
(lIter, ω, Δω) = (2, 4.7251679453883 - 1.8155253372388418im, 0.8101976035296421)
sqrt.(λ) = ComplexF64[2.026641508551375 - 0.08014162114490922im, 4.973144111882337 - 1.3650833563222078im]
(lIter, ω, Δω) = (3, 4.973144111882337 - 1.3650833563222078im, 0.23176795898507427)
sqrt.(λ) = ComplexF64[2.0180466148745695 - 0.07698005395849668im, 4.895070400595999 - 1.240745522195271im]
(lIter, ω, Δω) = (4, 4.895070400595999 - 1.240745522195271im, 0.06420186925066503)
sqrt.(λ) = ComplexF64[2.0154026963771967 - 0.07839660260944056im, 4.849171157170737 - 1.2343300834699218im]
(lIter, ω, Δω) = (5, 4.849171157170737 - 1.2343300834699218im, 0.020967783450161908)
sqrt.(λ) = ComplexF64[2.0152701617242745 - 0.07936

## Damped System with M, C, K

In [ ]:
function damped_system(idx, ω, M, C, K)
    
    ω_i = abs(imag(ω)) # using only the circular frequency part
    
    MFull = M - im*C/ω_i
    MFullReal = real.(MFull)
    MFullImag = imag.(MFull)

    Sol1 = M\K
    Sol2 = M\C

    
    CNew = -MFullImag * ω_i
    @show CNew
    @show C

    # Sol1 = MFullReal\K
    # Sol2 = MFullReal\ CNew
    
    sz = size(M,1)
    AFull = zeros(ComplexF64, 2sz, 2sz)
    AFull[1:sz, 1:sz] = zeros(sz, sz)
    AFull[1:sz, sz+1:end] = I(sz)
    AFull[sz+1:end, 1:sz] = -K
    AFull[sz+1:end, sz+1:end] = -C

    BFull = zeros(ComplexF64, 2sz, 2sz)
    BFull[1:sz, 1:sz] = I(sz)
    BFull[sz+1:end, sz+1:end] = M

    @show isreal(AFull)

    λ = LinearAlgebra.eigvals(AFull, BFull)
    V = LinearAlgebra.eigvecs(AFull, BFull)    

    return λ, V, det(AFull)
end

idx = 2

ω = 1.0 + im*1
lIter = 0    
Δω = 1 
αRelax = 0.8

ωₒ = ω
detMat = 0*im

# ωᵣ = αRelax * ω + (1 - αRelax) * ωₒ
# ωᵣ = real(ωᵣ)    
# λ, V, detMat = damped_system(idx, ωᵣ, M, C, K)     
# ω = λ[idx]
# @show detMat


while (Δω > 1e-5)
    
    ωᵣ = αRelax * ω + (1 - αRelax) * ωₒ
    # ωᵣ = real(ωᵣ)    

    ωₒ = ω     
    detMat_old = detMat

    λ, V, detMat = damped_system(idx, ωᵣ, M, C, K)     
    ω = λ[idx+1]
    @show λ 
    @show V 
    @show detMat
        
    # Δω = abs((ω - ωₒ)/ωₒ)
    Δω = abs((detMat - detMat_old)/(detMat_old + 1e-10))
    lIter += 1
    @show lIter, ω, Δω
end

println("Final ω: ", ω) 
println("Analytical ωd: ", ωd) 
println("Analytical ωd idx: ", diag(ωd)[idx]) 
println("Difference: ", imag(ω) - diag(ωd)[idx])

nothing

CNew = [0.4 0.0; 0.0 10.0]
C = [0.4 0.0; 0.0 10.0]
isreal(AFull) = true
λ = ComplexF64[-1.2500000000000007 - 4.841229182759271im, -1.2500000000000002 + 4.841229182759271im, -0.20000000000000004 - 1.9899748742132402im, -0.2 + 1.9899748742132395im]
V = ComplexF64[0.0 + 0.0im 0.0 + 0.0im -2.6258946917738622e-17 - 0.456626243421745im 0.45662624342174474 + 6.336962113686404e-18im; 2.1031363707000607e-17 + 0.16417047692613812im 0.16417047692613806 + 4.6410877236607996e-18im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im -0.908674751315651 + 0.09132524868434899im -0.091325248684349 + 0.908674751315651im; 0.7947869038423274 - 0.2052130961576727im -0.20521309615767275 + 0.7947869038423273im 0.0 + 0.0im 0.0 + 0.0im]
detMat = 400.0 + 0.0im
(lIter, ω, Δω) = (1, -0.20000000000000004 - 1.9899748742132402im, 4.0e12)
CNew = [0.4 0.0; 0.0 10.0]
C = [0.4 0.0; 0.0 10.0]
isreal(AFull) = true
λ = ComplexF64[-1.2500000000000007 - 4.841229182759271im, -1.2500000000000002 + 4.841229182759271im, -0.200000000